# Bulk-tagging scans by cluster, with structured tags

This notebook demonstrates `archival_structures.datasets.bulk_tagging.annotate_image_grid_with_tags`:
a paginated, checkbox-selectable grid of thumbnails (the same "whole cluster is X, uncheck the
exceptions" workflow used elsewhere) for tagging many scans at once -- but with a *structured*
tag builder (namespace dropdown, type/subtype comboboxes with vocabulary-driven suggestions, an
optional instance number) instead of a free-text box, writing straight into each scan's
`ScanAnnotation.tags`. See `archival_structures.datasets.vocabulary` (the "Vocabulary" guide in
the docs) for the full `namespace:type(:subtype)?(#N)?` grammar and its four namespaces.

The widget itself has no opinion on where `image_ids` come from -- it just needs a list of
thumbnail paths, e.g. one cluster's members from any clustering you've already run. This
notebook uses `archival_structures.analysis.page_layout_clustering.cluster_page_layouts`
(already demonstrated in `page-layout-clustering-demo.ipynb`) as a concrete, in-package source
of clusters, with one deliberate difference: it clusters whole, *unsplit* scans rather than
split verso/recto pages. `annotate_image_grid_with_tags` tags at the scan level (one set of
tags per whole image, not per page-within-an-opening), so clustering at the same granularity
keeps cluster membership in step with what gets tagged -- and keeps cluster membership keyed by
the original scan id, which for `NL-HaNA` is an exact match to the thumbnail filename
(see `archival_structures.datasets.annotations.parse_thumb_path`'s docstring for which archives
this resolves correctly for).

In [1]:
from pathlib import Path

import pagexml.parser as pagexml_parser

from archival_structures.analysis.page_layout_clustering import cluster_page_layouts
from archival_structures.datasets.bulk_tagging import annotate_image_grid_with_tags
from archival_structures.datasets.annotations import ANNOTATIONS_DIR, load_scan_tags

INSTITUTE = 'NL-HaNA'
ARCHIVE = 'NL-HaNA_2.10.50'
INVENTORY_NUM = 'NL-HaNA_2.10.50_1'
PAGEXML_DIR = Path('../../data/PageXML') / INSTITUTE / ARCHIVE / INVENTORY_NUM
THUMB_DIR = Path('../../data/thumbs') / INSTITUTE / ARCHIVE / INVENTORY_NUM

assert PAGEXML_DIR.exists(), PAGEXML_DIR
assert THUMB_DIR.exists(), THUMB_DIR

## Cluster a sample of scans by their layout

Same `GridPattern`-based clustering as `page-layout-clustering-demo.ipynb`, just run on whole
scans instead of split pages.

In [2]:
xml_paths = sorted(PAGEXML_DIR.glob('*.xml'))[:80]
scans = [pagexml_parser.parse_pagexml_file(str(xml_path)) for xml_path in xml_paths]
scans = [scan for scan in scans if len(scan.get_lines()) > 0]
print(f"{len(scans)} scans with at least one line")

clusters, grid_pattern = cluster_page_layouts(scans, unit_size=50, pattern_size=3, min_cluster_size=4)
print("\ncluster sizes:")
print(clusters.value_counts().sort_index())

78 scans with at least one line
done setting base grid points
took 0.0 seconds (total: 0.0)
done setting line grid_points for 78 docs, with 144 x and 119 y points
took 0.1 seconds (total: 0.1)


done computing patterns, 388 distinct patterns
took 1.4 seconds (total: 1.4)
done computing tf-idf of patterns
took 0.0 seconds (total: 1.5)

cluster sizes:
cluster
-1    62
 0     4
 1    12
Name: count, dtype: int64


## Pick a cluster and map it to thumbnail paths

`clusters` is a `pandas.Series` of cluster labels indexed by scan id (`-1` = noise/outliers).
For `NL-HaNA`, the thumbnail filename is an exact match to the scan id, so building the image
path list is just a join against `THUMB_DIR`.

In [3]:
CLUSTER_ID = clusters.value_counts().drop(-1, errors='ignore').idxmax()  # the largest real cluster
scan_ids = clusters[clusters == CLUSTER_ID].index.tolist()
image_ids = [str(THUMB_DIR / scan_id) for scan_id in scan_ids]
print(f"cluster {CLUSTER_ID}: {len(image_ids)} scans")
image_ids

cluster 1: 12 scans


['../../data/thumbs/NL-HaNA/NL-HaNA_2.10.50/NL-HaNA_2.10.50_1/NL-HaNA_2.10.50_1_0006.jpg',
 '../../data/thumbs/NL-HaNA/NL-HaNA_2.10.50/NL-HaNA_2.10.50_1/NL-HaNA_2.10.50_1_0010.jpg',
 '../../data/thumbs/NL-HaNA/NL-HaNA_2.10.50/NL-HaNA_2.10.50_1/NL-HaNA_2.10.50_1_0016.jpg',
 '../../data/thumbs/NL-HaNA/NL-HaNA_2.10.50/NL-HaNA_2.10.50_1/NL-HaNA_2.10.50_1_0018.jpg',
 '../../data/thumbs/NL-HaNA/NL-HaNA_2.10.50/NL-HaNA_2.10.50_1/NL-HaNA_2.10.50_1_0019.jpg',
 '../../data/thumbs/NL-HaNA/NL-HaNA_2.10.50/NL-HaNA_2.10.50_1/NL-HaNA_2.10.50_1_0037.jpg',
 '../../data/thumbs/NL-HaNA/NL-HaNA_2.10.50/NL-HaNA_2.10.50_1/NL-HaNA_2.10.50_1_0038.jpg',
 '../../data/thumbs/NL-HaNA/NL-HaNA_2.10.50/NL-HaNA_2.10.50_1/NL-HaNA_2.10.50_1_0046.jpg',
 '../../data/thumbs/NL-HaNA/NL-HaNA_2.10.50/NL-HaNA_2.10.50_1/NL-HaNA_2.10.50_1_0059.jpg',
 '../../data/thumbs/NL-HaNA/NL-HaNA_2.10.50/NL-HaNA_2.10.50_1/NL-HaNA_2.10.50_1_0060.jpg',
 '../../data/thumbs/NL-HaNA/NL-HaNA_2.10.50/NL-HaNA_2.10.50_1/NL-HaNA_2.10.50_1_0062.jpg',

## Bulk-tag the cluster

Pick a namespace, type, and (optionally) a subtype/instance number -- the suggestions update as
you go (type options depend on the namespace, subtype options depend on the type; `doctype:`
has no built-in suggestions since it's deliberately open, but offers anything already used
elsewhere under `data/annotations/` and grows as you tag). The live preview under the controls
shows the exact tag string that "Add to selected"/"Remove from selected" will use.

Selection persists across pages -- uncheck any scans that don't actually belong, build the tag,
then click "Add to selected" once. Saved immediately to each scan's `ScanAnnotation.tags`, no
separate import step.

In [4]:
annotate_image_grid_with_tags(image_ids, rows=3, cols=4, thumbnail_size=150)

## Verify what got tagged

`load_scan_tags` collects every scan-level tag under `data/annotations/` into a long-format
DataFrame -- filter to this inventory's scans to see what's been added so far.

In [5]:
tags_df = load_scan_tags(ANNOTATIONS_DIR)
tags_df[tags_df['scan_id'].isin(scan_ids)]

,scan_id,tag
